# Notebook 6: Evaluation Metrics & Statistical Testing

Compute tracking metrics and statistical significance tests.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from data_loader import SensorDataLoader
from attacks.camera_attacks import CameraAdversarialAttacker, AttackType
from defenses.defense_mechanisms import DefensePipeline, DefenseType
from evaluation.metrics import TrackingMetrics
from defenses.defense_mechanisms import StatisticalSignificance

%matplotlib inline

## 6.1 Load Data and Setup

In [ ]:
SCENARIO = 'scenario2'
DATA_PATH = f'../data/sensor_fusion_dataset/{SCENARIO}'

loader = SensorDataLoader(DATA_PATH)
detections = loader.load_all_detections()
ground_truth = loader.load_ground_truth()

# Create attacked and defended versions
attacker = CameraAdversarialAttacker(epsilon=0.05)
attacked = {sid: df.copy() for sid, df in detections.items()}
attacked[3] = attacker.attack_detections(detections[3].copy(), AttackType.FGSM, sensor_id=3)

pipeline = DefensePipeline()
defended = pipeline.defend_detections(attacked, DefenseType.TEMPORAL_CONSISTENCY, ground_truth)

metrics = TrackingMetrics()
print("Setup complete")

## 6.2 Compute Per-Sensor Metrics

In [ ]:
# Compute metrics for each condition
conditions = {
    'Benign': detections,
    'Attacked': attacked,
    'Defended': defended
}

all_metrics = {}
for condition_name, condition_dets in conditions.items():
    condition_metrics = {}
    for sensor_id in [1, 2, 3, 4]:
        m = metrics.compute_sensor_metrics(condition_dets[sensor_id], ground_truth, sensor_id)
        condition_metrics[sensor_id] = m
    all_metrics[condition_name] = condition_metrics

# Display as table
print("\nDetection Probability (%):")
print(f"{'Sensor':<10} {'Benign':<10} {'Attacked':<10} {'Defended':<10}")
print("-" * 40)
for sensor_id in [1, 2, 3, 4]:
    b = all_metrics['Benign'][sensor_id]['detection_probability'] * 100
    a = all_metrics['Attacked'][sensor_id]['detection_probability'] * 100
    d = all_metrics['Defended'][sensor_id]['detection_probability'] * 100
    print(f"{sensor_id:<10} {b:<10.1f} {a:<10.1f} {d:<10.1f}")

## 6.3 Metrics Comparison Visualization

In [ ]:
from visualization.plot_utils import plot_metrics_comparison

# Prepare metrics for plotting
metrics_dict = {
    'Benign': all_metrics['Benign'],
    'Attacked': all_metrics['Attacked'],
    'Defended': all_metrics['Defended']
}

fig = plot_metrics_comparison(metrics_dict, title='Metrics: Benign vs Attacked vs Defended')
plt.show()

## 6.4 Statistical Significance Testing

In [ ]:
# Run statistical tests
stats = StatisticalSignificance()

# Extract per-timestep detection probabilities for IR camera
def get_timestep_probs(dets, gt, sensor_id):
    probs = []
    times = sorted(dets[sensor_id]['time'].unique())
    for t in times:
        m = metrics.compute_sensor_metrics(dets[sensor_id], gt, sensor_id)
        probs.append(m['detection_probability'])
    return np.array(probs)

benign_probs = get_timestep_probs(detections, ground_truth, 3)
attacked_probs = get_timestep_probs(attacked, ground_truth, 3)
defended_probs = get_timestep_probs(defended, ground_truth, 3)

# Run tests
results = stats.compare_three_conditions(benign_probs, attacked_probs, defended_probs)

print("\nStatistical Significance Results (IR Camera):")
print(f"{'Comparison':<25} {'Mean Diff':<12} {'p-value':<12} {'Significant':<12} {'Cohen d':<10}")
print("-" * 75)
for comp, res in results.items():
    sig = "YES ***" if res['significant'] else "NO"
    print(f"{comp:<25} {res['mean_diff']:<12.4f} {res['p_value']:<12.4f} {sig:<12} {res['cohens_d']:<10.3f}")

## 6.5 Effect Size Visualization

In [ ]:
# Visualize effect sizes
comparisons = list(results.keys())
cohens_d = [results[c]['cohens_d'] for c in comparisons]
p_values = [results[c]['p_value'] for c in comparisons]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cohen's d
colors = ['green' if d < 0.2 else 'yellow' if d < 0.5 else 'orange' if d < 0.8 else 'red' for d in cohens_d]
bars = ax1.bar(comparisons, cohens_d, color=colors)
ax1.axhline(y=0.2, color='k', linestyle='--', alpha=0.5, label='Small')
ax1.axhline(y=0.5, color='k', linestyle='--', alpha=0.5, label='Medium')
ax1.axhline(y=0.8, color='k', linestyle='--', alpha=0.5, label='Large')
ax1.set_ylabel("Cohen's d")
ax1.set_title('Effect Size')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# p-values
ax2.bar(comparisons, p_values, color='steelblue')
ax2.axhline(y=0.05, color='r', linestyle='--', label='α = 0.05')
ax2.set_ylabel('p-value')
ax2.set_title('Statistical Significance')
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

plt.suptitle('Statistical Significance Testing Results', fontsize=14)
plt.tight_layout()
plt.show()

## 6.6 Per-Scenario Metrics Summary

In [ ]:
# Evaluate across multiple scenarios
scenarios = ['scenario2', 'scenario3', 'scenario4']
scenario_metrics = []

for scen in scenarios:
    path = f'../data/sensor_fusion_dataset/{scen}'
    ld = SensorDataLoader(path)
    dets = ld.load_all_detections()
    gt = ld.load_ground_truth()
    
    # Attack and defend
    att = CameraAdversarialAttacker(epsilon=0.05)
    att_dets = {sid: df.copy() for sid, df in dets.items()}
    att_dets[3] = att.attack_detections(dets[3].copy(), AttackType.FGSM, sensor_id=3)
    
    def_dets = pipeline.defend_detections(att_dets, DefenseType.TEMPORAL_CONSISTENCY, gt)
    
    # Metrics for IR camera
    m_benign = metrics.compute_sensor_metrics(dets[3], gt, 3)
    m_attacked = metrics.compute_sensor_metrics(att_dets[3], gt, 3)
    m_defended = metrics.compute_sensor_metrics(def_dets[3], gt, 3)
    
    scenario_metrics.append({
        'scenario': scen,
        'benign_detprob': m_benign['detection_probability'],
        'attacked_detprob': m_attacked['detection_probability'],
        'defended_detprob': m_defended['detection_probability'],
        'recovery': (m_defended['detection_probability'] / m_benign['detection_probability'] * 100) if m_benign['detection_probability'] > 0 else 0
    })

# Display
df_metrics = pd.DataFrame(scenario_metrics)
print(df_metrics.to_string(index=False))

## 6.7 Recovery Rate Across Scenarios

In [ ]:
plt.figure(figsize=(10, 6))

scenarios_names = [m['scenario'] for m in scenario_metrics]
benign_vals = [m['benign_detprob'] * 100 for m in scenario_metrics]
attacked_vals = [m['attacked_detprob'] * 100 for m in scenario_metrics]
defended_vals = [m['defended_detprob'] * 100 for m in scenario_metrics]

x = np.arange(len(scenarios_names))
width = 0.25

plt.bar(x - width, benign_vals, width, label='Benign', color='blue', alpha=0.7)
plt.bar(x, attacked_vals, width, label='Attacked', color='red', alpha=0.7)
plt.bar(x + width, defended_vals, width, label='Defended', color='green', alpha=0.7)

plt.xlabel('Scenario')
plt.ylabel('Detection Probability (%)')
plt.title('Defense Recovery Across Scenarios (IR Camera)')
plt.xticks(x, scenarios_names)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()